# Neural Network from Scratch
A feedforward neural network using only NumPy

In [20]:
import numpy as np
np.random.seed(42)

Generate a Dataset with 50K instances

In [21]:
X = np.random.randn(50000, 2)

# binary labels: 1 if x1 + x2 > 0, else 0
y = (X[:, 0] + X[:, 1] > 0).astype(int).reshape(-1, 1)


## Activation Functions

In [22]:

def sigmoid(z):
    # squashes values into the range (0, 1)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    # derivative of sigmoid used during backpropagation
    s = sigmoid(z)
    return s * (1 - s)


## Initialize Parameters

In [23]:

def initialize_parameters(input_dim, hidden_dim, output_dim):
    # small random weights help break symmetry
    W1 = np.random.randn(input_dim, hidden_dim) * 0.01
    b1 = np.zeros((1, hidden_dim))

    W2 = np.random.randn(hidden_dim, output_dim) * 0.01
    b2 = np.zeros((1, output_dim))

    return W1, b1, W2, b2


## Forward Propagation

In [24]:

def forward_propagation(X, W1, b1, W2, b2):
    # linear transformation for hidden layer
    Z1 = X @ W1 + b1

    # nonlinear activation
    A1 = sigmoid(Z1)

    # linear transformation for output layer
    Z2 = A1 @ W2 + b2

    # output probabilities
    A2 = sigmoid(Z2)

    return Z1, A1, Z2, A2


## Loss Function

In [25]:

def binary_cross_entropy(y, y_hat):
    # small epsilon avoids log(0)
    eps = 1e-8

    # average binary cross-entropy loss
    return -np.mean(
        y * np.log(y_hat + eps) +
        (1 - y) * np.log(1 - y_hat + eps)
    )


## Backpropagation

In [26]:

def backward_propagation(X, y, Z1, A1, A2, W2):
    m = X.shape[0]  # number of samples

    # gradient of loss w.r.t output pre-activation
    dZ2 = A2 - y

    # gradients for output layer parameters
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # backpropagate into hidden layer
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * sigmoid_derivative(Z1)

    # gradients for hidden layer parameters
    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2


## Training Loop

In [27]:

# network architecture
input_dim = 2
hidden_dim = 8
output_dim = 1

# training hyperparameters
lr = 0.1
epochs = 2000

# initialize weights and biases
W1, b1, W2, b2 = initialize_parameters(input_dim, hidden_dim, output_dim)

for epoch in range(epochs):
    # forward pass
    Z1, A1, Z2, A2 = forward_propagation(X, W1, b1, W2, b2)

    # compute loss
    loss = binary_cross_entropy(y, A2)

    # backward pass
    dW1, db1, dW2, db2 = backward_propagation(X, y, Z1, A1, A2, W2)

    # gradient descent parameter update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    # monitor training progress, not all 2000 but just a few epochs
    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")
        
# the training takes about a minute


Epoch 0, Loss: 0.6932
Epoch 200, Loss: 0.6892
Epoch 400, Loss: 0.5691
Epoch 600, Loss: 0.2857
Epoch 800, Loss: 0.1781
Epoch 1000, Loss: 0.1338
Epoch 1200, Loss: 0.1101
Epoch 1400, Loss: 0.0952
Epoch 1600, Loss: 0.0848
Epoch 1800, Loss: 0.0771


## Evaluation

In [28]:

# convert probabilities to binary predictions
preds = (A2 > 0.5).astype(int)

# classification accuracy
accuracy = np.mean(preds == y)
print("Accuracy:", accuracy)


Accuracy: 0.99962
